In [1]:
import pandas as pd
import numpy as np
import glob

# Análise da Energia

In [2]:
# topologia = [(0, 34), (25, 17), (25, 24), (8, 13), (8, 22), (14, 10), (14, 11), (28, 27), (28, 29), (28, 30), (2, 3), (2, 32), (5, 4), (9, 12), (23, 15), (23, 19), (23, 20), (18, 21), (6, 7), (2, 16), (25, 26), (33, 1), (33, 31), (34, 9), (9, 18), (18, 2), (2, 5), (5, 23), (23, 25), (25, 28), (28, 14), (14, 8), (8, 6), (6, 33), (33, 34)]
# central_link = (0, 34)
# # Criando o grafo a partir da lista de arestas
# G = nx.Graph()
# G.add_edges_from(topologia)

In [3]:
# #calculando a distancia dos nós
# nodes_distances = nx.shortest_path_length(G, source=0)
# del nodes_distances[0]
# sorted_nodes  = list(nodes_distances.keys())

In [4]:
# #calculando a distancia entre os links
# my_dict = {}
# for u, v in G.edges:
#     dst1 = nx.shortest_path_length(G,source=0,target=u)
#     dst2 = nx.shortest_path_length(G,source=0,target=v)
#     #print("edge:",u,v,", distancias: ", dst1, dst2)
#     dst = dst1-1 if dst1 > dst2 else dst2-1
#     my_dict[(u,v)] = dst
# order_dict = {}
# for i in sorted(my_dict, key = my_dict.get):
#     if  i == central_link:
#         continue
#     order_dict[i] = my_dict[i]

In [5]:
# sorted_links = []
# for key in order_dict.keys():
#     a = str(key).replace(' ','')
#     sorted_links.append((a))

In [6]:
dir_msf = glob.glob("..//..//results_cpu_50//sfc_on_alg_msf_50/*")
dir_ga = glob.glob("..//..//results_cpu_50//sfc_on_alg_ga_50/*")
dir_gr = glob.glob("..//..//results_cpu_50//sfc_on_alg_gr_50/*")

In [7]:
# dir_rod = glob.glob(f'..//..//results_cpu_50//sfc_on_alg_gr_50/*')

In [9]:
def get_data_alg(metric, dir_files):
    speed_files_alg = []
    for dir in dir_files:
        print(dir)
        if metric == "cache" or metric == "cpu":
            speed_files_alg.append(pd.read_csv(f"{dir}"))
        # if metric == "bandwidth" or metric == "edges_vnf":
        #     speed_files_alg.append(pd.read_csv(f'{dir}'))
    return speed_files_alg


cpu_msf = get_data_alg("cpu", dir_msf)
cpu_ga = get_data_alg("cpu", dir_ga)
cpu_gr = get_data_alg("cpu", dir_gr)

..//..//results_cpu_50//sfc_on_alg_msf_50\202403151928343810275990.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\202403151930467090843739.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\202403151930477044489130.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\202403151930486943081108.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\20240315193049686192716.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\202403151930507043235321.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\202403151930517229331017.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\202403151930527061984756.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\202403151930537018408232.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\202403151930547036464281.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\202403161343372800614251.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\202403161343380215953641.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\202403161343390150166856.csv
..//..//results_cpu_50//sfc_on_alg_msf_50\202403161343400077701171.csv
..//../

In [16]:
def process_cpu_data(cpu_dir):
    cpu_dfs = []
    time_ranges = []
    for cpu_df in cpu_dir:
        referencia = cpu_df["timestamp"].iloc[0]
        # Convertendo todos os valores para uma contagem de segundos a partir do valor de referência
        cpu_df["tempo"] = cpu_df["timestamp"] - referencia
        cpu_df["tempo"] = cpu_df["tempo"].astype(int)
        cpu_df = cpu_df.drop(columns="timestamp")

        df_mean = cpu_df.groupby("tempo").mean().reset_index()

        # Determina o valor máximo de tempo no DataFrame
        tempo_range = df_mean["tempo"].max()

        # Reindexa o DataFrame para incluir todos os tempos no intervalo desejado
        df_mean = df_mean.set_index("tempo").reindex(range(tempo_range + 1))
        df_mean = df_mean.ffill()

        # Reseta o índice do DataFrame
        df_mean = df_mean.reset_index()

        # Verifica se o DataFrame tem o tamanho mínimo requerido antes de adicionar à lista
        if len(df_mean) >= 1000:
            cpu_dfs.append(df_mean)
            time_ranges.append(tempo_range)
            # processing_nodes = np.array([25, 8, 14, 28, 2, 5, 9, 23, 18, 6, 33, 34, "tempo"])
            # cpu_df = cpu_df.loc[cpu_df.index.isin(processing_nodes)]

    print(time_ranges)
    # Considera o menor intervalo de tempo entre todos os DataFrames
    min(time_ranges)

    # Concatena os DataFrames, limitando a 900 segundos, e calcula a média para cada índice de tempo
    cpu = pd.concat([cpu_df.iloc[0:1000] for cpu_df in cpu_dfs])
    cpu = cpu.groupby(cpu.index).mean()
    cpu["tempo"] = cpu["tempo"].astype(int)
    cpu.set_index("tempo", inplace=True, drop=True)

    return cpu


cpu_msf_df = process_cpu_data(cpu_msf)
cpu_ga_df = process_cpu_data(cpu_ga)
cpu_gr_df = process_cpu_data(cpu_gr)

[1012, 1014, 1013, 1013, 1012, 1013, 1013, 1013, 1012, 1012, 1017, 1170, 1089, 1192, 1186, 1169, 1191, 1190, 1185, 1017, 1172, 1167, 1172, 1018, 1160, 1176, 1017, 1109, 1158, 1012]
[1013, 1654, 1671, 1655, 1656, 1455, 1681, 1650, 1678, 1662, 1679, 1116, 1674, 1638, 1656, 1657, 1639, 1668, 1022, 1022, 1660, 1013, 1013, 1012, 1012, 1012, 1013, 1012, 1013, 1013]
[1010, 1010, 1010, 1010, 1010, 1010, 1010, 1010, 1010, 1010, 1012, 1011, 1012, 1012, 1012, 1012, 1012, 1012, 1012, 1012, 1012, 1012, 1012, 1012, 1012, 1012, 1012, 1013, 1012, 1012]


In [20]:
# Supondo que 'process_cpu_data' seja uma função que você já tem para processar seus dados de CPU
# e que cpu_dp, cpu_ga, cpu_rod são seus DataFrames originais de dados de CPU para cada algoritmo.

# Aplicando o agrupamento e média como descrito anteriormente
def group_and_average(df, step=50):
    # Criando um novo índice que representa os grupos de tempo
    grouped_index = df.index // step * step
    # Agrupando e calculando a média
    return df.groupby(grouped_index).mean().T


# Processando e agrupando os dados de CPU para cada algoritmo
df_grouped_msf = group_and_average(cpu_msf_df)
df_grouped_ga = group_and_average(cpu_ga_df)
df_grouped_gr = group_and_average(cpu_gr_df)

In [25]:
processing_nodes = np.array(["25", "8", "14", "28", "2", "5", "9", "23", "18", "6", "33", "34"])

In [28]:
df_grouped_msf = df_grouped_msf.loc[df_grouped_msf.index.isin(processing_nodes)]
df_grouped_ga = df_grouped_ga.loc[df_grouped_ga.index.isin(processing_nodes)]
df_grouped_gr = df_grouped_gr.loc[df_grouped_gr.index.isin(processing_nodes)]

In [31]:
import plotly.graph_objects as go

# Encontrando os valores mínimos e máximos globais entre todos os DataFrames para definir a escala de cores
min_value = min(df_grouped_msf.min().min(), df_grouped_gr.min().min())
max_value = max(df_grouped_msf.max().max(), df_grouped_gr.max().max())


# Função para criar o mapa de calor
def plot_heatmap(df, title):
    fig = go.Figure(
        data=go.Heatmap(
            z=df.values,
            x=df.columns,
            y=df.index,
            zmin=min_value,  # Define a mesma escala mínima para todos os mapas de calor
            zmax=max_value,  # Define a mesma escala máxima para todos os mapas de calor
            colorbar=dict(title="CPU Usage"),
            texttemplate="%{z:.2f}",
            textfont={"size": 10},
        )
    )

    fig.update_layout(
        title=title,
        yaxis_title="Server",
        xaxis_title="Time Interval",
        height=700,  # Configura a altura do gráfico
    )
    fig.show()


# Plotando mapas de calor para cada algoritmo com a mesma escala de cores
plot_heatmap(df_grouped_msf, "MSF Algorithm Average CPU Usage")
plot_heatmap(df_grouped_ga, "GA Algorithm Average CPU Usage")
plot_heatmap(df_grouped_gr, "Guided Algorithm Average CPU Usage")

# Gerando o dataframe de energia

### Possibilidade de cálculos

In [36]:
processing_nodes = np.array(["25", "8", "14", "28", "2", "5", "9", "23", "18", "6", "33", "34"])

In [39]:
cpu_msf_df = cpu_msf_df[processing_nodes]
cpu_ga_df = cpu_ga_df[processing_nodes]
cpu_gr_df = cpu_gr_df[processing_nodes]

In [40]:
def calculate_total_power(P_base, P_idle, P_max, U):
    """
    Calcula o consumo total de energia com base no uso da CPU.

    Parâmetros:
    - P_base: Consumo de energia base do servidor sem carga de CPU.
    - P_idle: Consumo de energia da CPU no estado ocioso.
    - P_max: Consumo de energia da CPU com 100% de carga.
    - U: Percentual de uso da CPU (0 a 100).

    Retorna:
    - Consumo total de energia em watts.
    """
    return P_base + (P_max - P_idle) * (U / 100) + P_idle


# Parâmetros de energia
P_base = 50  # Consumo de energia base em watts
P_idle = 20  # Consumo de energia da CPU em estado ocioso em watts
P_max = 100  # Consumo de energia da CPU a 100% de carga em watts

# Aplicando a função de cálculo de energia ao DataFrame
df_energy_msf = cpu_msf_df.applymap(lambda U: calculate_total_power(P_base, P_idle, P_max, U))
df_energy_ga = cpu_ga_df.applymap(lambda U: calculate_total_power(P_base, P_idle, P_max, U))
df_energy_gr = cpu_gr_df.applymap(lambda U: calculate_total_power(P_base, P_idle, P_max, U))

# Supondo que df_energy_dp, df_energy_ga, df_energy_rod sejam os DataFrames de energia
# Vamos calcular a média do consumo de energia da rede inteira ao longo do tempo para cada algoritmo
average_energy_msf = df_energy_msf.mean(axis=1)
average_energy_ga = df_energy_ga.mean(axis=1)
average_energy_gr = df_energy_gr.mean(axis=1)

# Preparando os dados para plotagem
time_index = range(
    len(average_energy_msf)
)  # Assumindo que todos os DataFrames têm o mesmo comprimento

# Criando um DataFrame para facilitar a plotagem com Plotly
df_comparison = pd.DataFrame(
    {
        "Time": time_index,
        "MSF": average_energy_msf.values,
        "Genetic": average_energy_ga.values,
        "Guided": average_energy_gr.values,
    }
)

# Plotando o gráfico de comparação usando Plotly

fig = go.Figure()

# Adicionando cada série de dados
fig.add_trace(
    go.Scatter(x=df_comparison["Time"], y=df_comparison["MSF"], mode="lines", name="MSF")
)
fig.add_trace(
    go.Scatter(x=df_comparison["Time"], y=df_comparison["Genetic"], mode="lines", name="GA")
)
fig.add_trace(
    go.Scatter(x=df_comparison["Time"], y=df_comparison["Guided"], mode="lines", name="Guided")
)

fig.update_layout(
    title="Comparison of Energy Consumption by Algorithm Over Time",
    xaxis_title="Time",
    yaxis_title="Average Energy Consumption (Watts)",
    legend_title="Algorithm",
)

# Exibindo o gráfico
fig.show()

C:\Users\rodri\AppData\Local\Temp\ipykernel_16440\3777142629.py:22: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

C:\Users\rodri\AppData\Local\Temp\ipykernel_16440\3777142629.py:23: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

C:\Users\rodri\AppData\Local\Temp\ipykernel_16440\3777142629.py:24: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



In [17]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


# Atualizando a função de cálculo de consumo com a lógica revisada
def calcular_consumo_revisado(cpu_percent):
    consumo_base = 10  # Watts, assumindo um consumo base para baixo uso ou standby
    if cpu_percent >= 17.27:
        return 0.4342 * cpu_percent + 10.501
    else:
        return consumo_base


# Exemplo de aplicação (substitua pelos seus DataFrames reais)
df_energy_dp = cpu_dp_df.applymap(calcular_consumo_revisado)
df_energy_ga = cpu_ga_df.applymap(calcular_consumo_revisado)
df_energy_rod = cpu_rod_df.applymap(calcular_consumo_revisado)

# Calcular a média do consumo de energia da rede inteira ao longo do tempo para cada algoritmo
average_energy_dp = df_energy_dp.mean(axis=1)
average_energy_ga = df_energy_ga.mean(axis=1)
average_energy_rod = df_energy_rod.mean(axis=1)

time_index = range(
    len(average_energy_dp)
)  # Assumindo que todos os DataFrames têm o mesmo comprimento


# Criando um DataFrame para facilitar a plotagem com Plotly
df_comparison = pd.DataFrame(
    {
        "Time": time_index,
        "DP Algorithm": average_energy_dp.values,
        "GA Algorithm": average_energy_ga.values,
        "ROD Algorithm": average_energy_rod.values,
    }
)

# Plotando o gráfico de comparação usando Plotly
import plotly.graph_objects as go

fig = go.Figure()

# Adicionando cada série de dados
fig.add_trace(
    go.Scatter(
        x=df_comparison["Time"], y=df_comparison["DP Algorithm"], mode="lines", name="DP Algorithm"
    )
)
fig.add_trace(
    go.Scatter(
        x=df_comparison["Time"], y=df_comparison["GA Algorithm"], mode="lines", name="GA Algorithm"
    )
)
fig.add_trace(
    go.Scatter(
        x=df_comparison["Time"],
        y=df_comparison["ROD Algorithm"],
        mode="lines",
        name="ROD Algorithm",
    )
)

fig.update_layout(
    title="Comparison of Energy Consumption by Algorithm Over Time",
    xaxis_title="Time",
    yaxis_title="Average Energy Consumption (Watts)",
    legend_title="Algorithm",
)

# Exibindo o gráfico
fig.show()

NameError: name 'cpu_dp_df' is not defined